# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HarisBinHabib/Internship-Repo/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [16]:
import os
if not os.path.exists("/content/repo"):
    !git clone https://github.com/HarisBinHabib/Internship-Repo.git /content/repo
%cd /content/repo

!pip install duckdb --quiet

/content/repo


In [17]:
import os, getpass
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

In [18]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':       f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':       f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample': f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':    f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

# Only count the small dimension tables — skip full counts on the huge fact tables
small_tables = ['dim_clients', 'dim_content']
for name in small_tables:
    n = con.sql(f"SELECT COUNT(*) FROM {TABLES[name]}").fetchone()[0]
    print(f'{name:22} {n:>12,} rows')

print("fact_daily and fact_daily_sample skipped (full count too slow) — "
      "verified instead via the windowed queries in Section 3.")

dim_clients                     104 rows
dim_content                 519,606 rows
fact_daily and fact_daily_sample skipped (full count too slow) — verified instead via the windowed queries in Section 3.


In [19]:
import os
if not os.path.exists("/content/repo"):
    !git clone https://github.com/HarisBinHabib/Internship-Repo.git /content/repo
%cd /content/repo

!pip install duckdb --quiet

/content/repo


## 1. Unit of analysis + time window

**One row = one (client, content) pair's aggregated performance**, summarized over a
30-day prior feature window ending at a fixed decision point, using March 2026 as my
mid-panel iteration month.

**Anchor date:** `2026-03-31`.
**Decision point:** `2026-03-01` (30 days before the anchor).
**Feature window:** `2026-01-30` to `2026-03-01` (the 30 days strictly BEFORE the decision
point — everything known at decision time).
**Label window:** `2026-03-01` to `2026-03-31` (the 30 days AFTER the decision point — what
actually happened next, used only to build the label, never as a feature).

This is verified below with a grain check on the raw `fact_content_daily_performance` table.

In [20]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Verify the RAW table's grain: one row per report_date x client x content
grain_check = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
    FROM {TABLES['fact_daily']}
    WHERE report_date BETWEEN DATE '2026-01-30' AND DATE '2026-03-31'
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

print(f"Rows violating the grain (should be empty): {len(grain_check)}")
grain_check


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows violating the grain (should be empty): 0


,report_date,client_hash_id,content_hash_id,c


## 2. Fields: feature / label / context / excluded

**Feature** (knowable before the decision point, `2026-03-01`):
- `imp_prev30` — GSC impressions in the 30 days before the decision point
- `clk_prev30` — GSC clicks in the same prior window
- `pos_prev30` — average GSC position in the same prior window
- `ctr_prev30` — clicks/impressions in the same prior window
- `client_tenure_days` — days since the client's `gsc_data_start` (a static dimension fact)

**Label / proxy:**
- `declined_next30` — 1 if `imp_last30 < 0.70 * imp_prev30`, else 0. Built from the 30 days
  AFTER the decision point. This is a future-observed proxy, not a product-rule label.

**Context** (never features — join/group only):
- `client_hash_id`, `content_hash_id`

**Excluded, with why:**
- `imp_last30`, `clk_last30`, `pos_last30` — these come from the same window that defines the
  label; using them as features would hand the model the answer directly (the leakage trap in
  Section 3 shows exactly why).
- `keyword_hash_id`, `url_hash_id`, raw query/URL fields — grouping/dedup context only, never
  model inputs, per the data contract's own field-type rules.
- FlyRank product decision fields (`health_score`, `priority_score`, `action_type`) — not
  present in this warehouse release at all; excluded by design, not by my choice.

In [21]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# No query needed here — this section is the classification itself.
# The three verification queries for these claims live in Section 3.


## 3. Verify it with queries (grain, counts, missing values, windows) + features + the trap

Three proof queries, a five-feature frame with availability reasoning, then the deliberate
leakage trap.

In [22]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
slice_stats = con.sql(f"""
    SELECT COUNT(*) AS row_count, MIN(report_date) AS min_date, MAX(report_date) AS max_date
    FROM {TABLES['fact_daily']}
    WHERE report_date BETWEEN DATE '2026-01-30' AND DATE '2026-03-31'
""").df()

print("March 2026 slice (plus its 30-day lookback):")
slice_stats

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

March 2026 slice (plus its 30-day lookback):


,row_count,min_date,max_date
0,17717009,2026-01-30,2026-03-31


In [23]:
availability = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available_rows
    FROM {TABLES['fact_daily']}
    WHERE report_date BETWEEN DATE '2026-01-30' AND DATE '2026-03-31'
""").df()

print("How many rows actually have usable GA4 data (IS TRUE filter):")
availability
availability['ga4_available_pct'] = (
    availability['ga4_available_rows'] / availability['total_rows'] * 100
)
availability

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

How many rows actually have usable GA4 data (IS TRUE filter):


,total_rows,ga4_available_rows,ga4_available_pct
0,17717009,567080.0,3.200766


In [24]:
features = con.sql(f"""
    WITH anchor AS (SELECT DATE '2026-03-31' AS d),
    windowed AS (
        SELECT
            f.client_hash_id, f.content_hash_id,
            SUM(CASE WHEN f.report_date > a.d - INTERVAL 60 DAY
                      AND f.report_date <= a.d - INTERVAL 30 DAY
                      THEN f.gsc_impressions ELSE 0 END) AS imp_prev30,
            SUM(CASE WHEN f.report_date > a.d - INTERVAL 60 DAY
                      AND f.report_date <= a.d - INTERVAL 30 DAY
                      THEN f.gsc_clicks ELSE 0 END) AS clk_prev30,
            AVG(CASE WHEN f.report_date > a.d - INTERVAL 60 DAY
                      AND f.report_date <= a.d - INTERVAL 30 DAY
                      THEN f.gsc_avg_position END) AS pos_prev30,
            SUM(CASE WHEN f.report_date > a.d - INTERVAL 30 DAY
                      THEN f.gsc_impressions ELSE 0 END) AS imp_last30
        FROM {TABLES['fact_daily']} f, anchor a
        WHERE f.report_date > a.d - INTERVAL 60 DAY
        GROUP BY 1, 2
        HAVING imp_prev30 >= 100
    )
    SELECT w.*, c.gsc_data_start
    FROM windowed w
    LEFT JOIN {TABLES['dim_clients']} c ON w.client_hash_id = c.client_hash_id
""").df()

import pandas as pd
features['ctr_prev30'] = features['clk_prev30'] / features['imp_prev30']
features['client_tenure_days'] = (
    pd.Timestamp('2026-03-01') - pd.to_datetime(features['gsc_data_start'])
).dt.days
features['declined_next30'] = (features['imp_last30'] < 0.70 * features['imp_prev30']).astype(int)

print(f"{len(features):,} (client, content) pairs with enough prior-30-day history")
features[['imp_prev30', 'clk_prev30', 'pos_prev30', 'ctr_prev30',
          'client_tenure_days', 'declined_next30']].head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

82,025 (client, content) pairs with enough prior-30-day history


,imp_prev30,clk_prev30,pos_prev30,ctr_prev30,client_tenure_days,declined_next30
0,166.0,1.0,4.485106,0.006024,383,0
1,8430.0,8.0,5.045716,0.000949,383,0
2,357.0,0.0,7.407040,0.000000,383,0
3,7783.0,35.0,1.704329,0.004497,383,0
4,173.0,1.0,8.770291,0.005780,383,0


**Five features, each knowable before the decision point (2026-03-01):**
1. `imp_prev30` — measured entirely within the prior 30-day window, before the decision point.
2. `clk_prev30` — same prior window, same reasoning.
3. `pos_prev30` — averaged only over the prior window.
4. `ctr_prev30` — derived purely from `clk_prev30` and `imp_prev30`, both already safe.
5. `client_tenure_days` — a static client fact (`gsc_data_start`), known from before tracking
   even began; can't leak the future.

In [25]:
from sklearn.metrics import roc_auc_score

# --- THE TRAP: add ONE label-derived column on purpose ---

# Honest: can a pre-decision feature predict the label at all?
honest_auc = roc_auc_score(features['declined_next30'], features['pos_prev30'])
print(f"Honest quick score (AUC using only pre-decision pos_prev30): {honest_auc:.3f}")

# Now sneak in a feature built from the SAME window that defines the label —
# this ratio directly reconstructs the 0.70 threshold used to build declined_next30
features['leaked_feature'] = features['imp_last30'] / features['imp_prev30']
leaked_auc = roc_auc_score(features['declined_next30'], -features['leaked_feature'])
print(f"Leaked quick score (AUC using imp_last30/imp_prev30, the future window): {leaked_auc:.3f}")
print("\nThat jump isn't a better model — it's the label looking at itself.")

# Delete the leaked column, keep only the honest number
features = features.drop(columns=['leaked_feature'])
print(f"\nHonest number kept: AUC = {honest_auc:.3f}")

Honest quick score (AUC using only pre-decision pos_prev30): 0.630
Leaked quick score (AUC using imp_last30/imp_prev30, the future window): 1.000

That jump isn't a better model — it's the label looking at itself.

Honest number kept: AUC = 0.630


## 4. Data limits

**Named limitation: this slice sits on an unbalanced panel.** Client tracking histories don't
start together — `dim_clients.gsc_data_start` varies a lot per client. Some clients in this
March 2026 slice may not have had 60 full days of tracking before the anchor date, which
silently shrinks their usable history to less than the full window I assumed. Rows before a
client's `ga4_data_start` also carry GSC data only, with GA4 columns zero-filled — the
availability check in Section 3 shows this isn't rare. Any client whose tracking started
inside my 60-day lookback would have an artificially thin `imp_prev30`, not a genuinely low
one — treating those two cases the same would be a mistake I'm flagging on purpose rather
than fixing here.

In [26]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Show how many clients in this feature frame have thin history within the lookback window
thin_history = features.merge(
    con.sql(f"SELECT client_hash_id, gsc_data_start FROM {TABLES['dim_clients']}").df(),
    on='client_hash_id', how='left'
)
cutoff = pd.Timestamp('2026-01-30')
thin_history['thin_history'] = pd.to_datetime(thin_history['gsc_data_start_y']) > cutoff
print(f"Rows with less than the full 60-day lookback available: {thin_history['thin_history'].sum()} "
      f"of {len(thin_history)} ({thin_history['thin_history'].mean()*100:.1f}%)")


Rows with less than the full 60-day lookback available: 7876 of 82025 (9.6%)


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.